In [ ]:
# ============================================================
# SALUD DEL PROYECTO — versión actualizada
# Verifica todos los archivos y hace chequeos básicos
# ============================================================
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# --- Setup ---
RAIZ = Path(r'C:\Users\HP\Documents\Economy\economia_c')
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

RAW = RAIZ / 'data' / 'raw'
PROC = RAIZ / 'data' / 'processed'
INF = RAIZ / 'informe'
FIG = INF / 'figuras'

print('=' * 80)
print('🏥 SALUD DEL PROYECTO — ECONOMÍA DEL CONOCIMIENTO LA RIOJA')
print('=' * 80)

# ============================================================
# 1. ARCHIVOS RAW ESENCIALES
# ============================================================
print('\n📁 1. ARCHIVOS RAW ESENCIALES')
print('-' * 80)

RAW_ESENCIALES = {
    'provinciales_serie_empleo_trimestral_4dig_7.xlsx': 'Empleo EC por CIIU 4 dígitos',
    'provinciales_serie_empleo_anual_3.xlsx':           'Empleo por sectores (T5)',
    'provinciales_serie_empresas3_2.xlsx':              'Empresas por CIIU 4 dígitos',
    'provinciales_serie_remuneraciones_mensual_4dig_7.xlsx': 'Salarios por CIIU',
    'SerieWeb_EMPLEO_PRIVADO_FORMAL_provincias.xlsx':   'Empleo formal mensual',
    'SerieWeb_SALARIO_PRIVADO_FORMAL_provincias.xlsx':  'Salario formal mensual',
}

raw_ok = 0
for archivo, desc in RAW_ESENCIALES.items():
    ruta = RAW / archivo
    if ruta.exists():
        size_kb = ruta.stat().st_size / 1024
        print(f'   ✅ {archivo:<55} ({size_kb:>7.0f} KB)')
        raw_ok += 1
    else:
        print(f'   ❌ {archivo:<55} FALTA')

print(f'\n   Total: {raw_ok}/{len(RAW_ESENCIALES)} archivos raw OK')

# ============================================================
# 2. ARCHIVOS PROCESSED ACTIVOS
# ============================================================
print('\n📁 2. ARCHIVOS PROCESSED ACTIVOS')
print('-' * 80)

PROC_ESENCIALES = {
    'empleo_ec_2025_trimestral.csv':    'Empleo EC 2025 por CIIU',
    'empleo_ec_por_ciiu_2022_2025.csv': 'Serie Empleo EC 2022-2025',
    'empresas_ec_rioja_2022_2025.xlsx': 'Empresas EC',
    'prima_salarial_ec.xlsx':           'Prima salarial',
    'empleo_formal_ia_mensual.csv':     'Empleo formal mensual',
    'intensidad_provincias.csv':        'Intensidad 9 provincias',
    'Empleo_EC_2025_La_Rioja.xlsx':     'Excel auditable',
}

proc_ok = 0
for archivo, desc in PROC_ESENCIALES.items():
    ruta = PROC / archivo
    if ruta.exists():
        size_kb = ruta.stat().st_size / 1024
        print(f'   ✅ {archivo:<45} ({size_kb:>7.1f} KB) — {desc}')
        proc_ok += 1
    else:
        print(f'   ❌ {archivo:<45} FALTA — {desc}')

print(f'\n   Total: {proc_ok}/{len(PROC_ESENCIALES)} archivos processed OK')

# ============================================================
# 3. VERIFICACIÓN DE DATOS CLAVE
# ============================================================
print('\n📊 3. VERIFICACIÓN DE DATOS CLAVE (2025)')
print('-' * 80)

try:
    # --- Empleo EC 2025 ---
    emp = pd.read_csv(PROC / 'empleo_ec_2025_trimestral.csv')
    total_ec = emp['Promedio 2025'].sum()
    
    # Núcleo informático
    nucleo_codes = [7220, 7250, 7290]
    nucleo = emp[emp['CIIU'].isin(nucleo_codes)]['Promedio 2025'].sum()
    
    # Proxies
    proxy_codes = [7421, 7499]
    proxies = emp[emp['CIIU'].isin(proxy_codes)]['Promedio 2025'].sum()
    
    print(f'\n   📌 Empleo EC 2025:')
    print(f'      Total EC (8 CIIU):       {total_ec:>7.1f} puestos   (esperado ~521)')
    print(f'      Núcleo informático:      {nucleo:>7.1f} puestos   (esperado ~63.8)')
    print(f'      Proxies:                 {proxies:>7.1f} puestos   (esperado ~457.6)')
    print(f'      Núcleo como % del EC:    {nucleo/total_ec*100:>7.1f}%         (esperado ~12.2%)')
except Exception as e:
    print(f'   ⚠️  Error al leer empleo: {e}')

try:
    # --- Empresas EC ---
    empre = pd.read_excel(PROC / 'empresas_ec_rioja_2022_2025.xlsx')
    print(f'\n   📌 Empresas EC:')
    print(f'      Archivo:                 {empre.shape[0]} filas × {empre.shape[1]} columnas')
    print(f'      Columnas:                {empre.columns.tolist()[:5]}')
except Exception as e:
    print(f'   ⚠️  Error al leer empresas: {e}')

try:
    # --- Prima salarial ---
    prima = pd.read_excel(PROC / 'prima_salarial_ec.xlsx')
    print(f'\n   📌 Prima salarial:')
    print(f'      Archivo:                 {prima.shape[0]} filas × {prima.shape[1]} columnas')
    print(f'      Columnas:                {prima.columns.tolist()[:5]}')
except Exception as e:
    print(f'   ⚠️  Error al leer prima: {e}')

try:
    # --- Intensidad ---
    inten = pd.read_csv(PROC / 'intensidad_provincias.csv')
    print(f'\n   📌 Intensidad empresarial:')
    print(f'      Provincias:              {len(inten)}')
    print(f'      Columnas:                {inten.columns.tolist()}')
except Exception as e:
    print(f'   ⚠️  Error al leer intensidad: {e}')

# ============================================================
# 4. INFORMES Y DASHBOARD
# ============================================================
print('\n📁 4. INFORMES Y DASHBOARD')
print('-' * 80)

# Informe vigente
informe_qmd = INF / 'informe_empleo_ec.qmd'
if informe_qmd.exists():
    size_kb = informe_qmd.stat().st_size / 1024
    print(f'   ✅ Informe vigente: informe_empleo_ec.qmd ({size_kb:.1f} KB)')
else:
    print(f'   ❌ Informe vigente NO encontrado')

# Figuras usadas
figuras_usadas = [
    'F41_empleo_ec_2025_por_ciiu.png',
    'empleo_ec_2022_2025_por_ciiu.png',
    'F33_salarios_ciiu.png',
    'F34_prima_salarial.png',
    'F36_empleo_share.png',
    'F37_intensidad.png',
]
figs_ok = sum(1 for f in figuras_usadas if (FIG / f).exists())
print(f'   Figuras usadas por el informe: {figs_ok}/{len(figuras_usadas)}')

# Dashboard
dash = RAIZ / 'dashboard' / 'index.html'
if dash.exists():
    size_kb = dash.stat().st_size / 1024
    print(f'   ✅ Dashboard: index.html ({size_kb:.1f} KB)')
else:
    # Buscar en otros lugares
    candidatos = list(RAIZ.rglob('index.html'))
    candidatos = [c for c in candidatos if '.venv' not in str(c)]
    if candidatos:
        print(f'   ⚠️  Dashboard encontrado en: {candidatos[0].relative_to(RAIZ)}')
    else:
        print(f'   ❌ Dashboard NO encontrado')

# ============================================================
# 5. MÓDULOS EN src/
# ============================================================
print('\n📁 5. MÓDULOS EN src/')
print('-' * 80)

SRC = RAIZ / 'src'
modulos_clave = [
    'empresas_ec.py',
    'prima_salarial.py',
    'recaudacion_ec.py',
    'empleo.py',
    'benchmark_intensidad.py',
]

for mod in modulos_clave:
    ruta_mod = SRC / mod
    if ruta_mod.exists():
        size_kb = ruta_mod.stat().st_size / 1024
        print(f'   ✅ {mod:<30} ({size_kb:>6.1f} KB)')
    else:
        print(f'   ❌ {mod:<30} FALTA')

# ============================================================
# 6. RESUMEN FINAL
# ============================================================
print('\n' + '=' * 80)
print('📋 RESUMEN')
print('=' * 80)

total_checks = len(RAW_ESENCIALES) + len(PROC_ESENCIALES) + len(modulos_clave)
total_ok = raw_ok + proc_ok

print(f'\n   Archivos raw:           {raw_ok}/{len(RAW_ESENCIALES)} OK')
print(f'   Archivos processed:     {proc_ok}/{len(PROC_ESENCIALES)} OK')
print(f'   Informe vigente:        {"✅ OK" if informe_qmd.exists() else "❌ FALTA"}')
print(f'   Dashboard:              {"✅ OK" if dash.exists() else "⚠️  Revisar"}')
print(f'   Módulos src/:           {sum(1 for m in modulos_clave if (SRC/m).exists())}/{len(modulos_clave)} OK')

if raw_ok == len(RAW_ESENCIALES) and proc_ok == len(PROC_ESENCIALES):
    print('\n   ✅ PROYECTO SALUDABLE')
else:
    print('\n   ⚠️  Faltan archivos. Revisar arriba.')

print('=' * 80)